# Globular clusters in a live Milky Way: Pal 5

Pal 5 on its observed orbit (Baumgardt's catalogue + Gaia EDR3 proper motions), integrated back in time
and then forward to the present **in a live N-body Milky Way**, with jaccpot's `ProductionRunner` and
nornax's hybrid Hermite clusters (`ClustersRunnerScheme`): the cluster's own stars are a direct-sum
Hermite-4 system, the galaxy an FMM, and stars beyond 2 r_J are stripped into the galaxy's rows.

The sanity check is the same cluster as a plain N-body system in Odisseo's **static** Milky Way potential,
fitted to the live model: the stream should agree to within the live-vs-static difference.

Every run is a script in `notebooks/gc_in_mw/` writing to `$ODISSEO_GC_RUNS` (resumable: rerun the same
command); this notebook only launches them and plots. See `docs/source/globular_clusters.md`.

**Modes.** `SMALL = True` (default) runs a CPU version (galaxy 5000, M = 64, 2 Myr; ~10-20 min, compile-dominated): it
tests the pipeline, not the physics. `SMALL = False` is the GPU demo whose figures are committed below
(galaxy 1e7, M = 4096, 2 Gyr; run the scripts on a GPU, e.g. through `on-free-card`).
`RUN = False` only plots existing outputs.

In [ ]:
import os, subprocess, sys, json
from IPython.display import Image, display

SMALL = bool(int(os.environ.get("GC_SMALL", "1")))   # CPU small mode
RUN = bool(int(os.environ.get("GC_RUN", "1")))       # launch the scripts (else: plot existing outputs)
HERE = os.path.abspath("gc_in_mw") if os.path.isdir("gc_in_mw") else os.path.abspath(".")
REPO = os.path.abspath(os.path.join(HERE, "..", ".."))
RUNS = os.environ.setdefault("ODISSEO_GC_RUNS", os.path.join(HERE, "runs"))
DATA = os.environ.setdefault("ODISSEO_GC_DATA", os.path.join(RUNS, "catalogue"))
JAX_PY = os.environ.get("GC_JAX_PYTHON", sys.executable)      # jax >= 0.11.2 + jaccpot + nornax
AGAMA_PY = os.environ.get("GC_AGAMA_PYTHON", sys.executable)  # agama (ICs, prepare)
PREFIX = os.environ.get("GC_GPU_PREFIX", "").split()          # e.g. "on-free-card 20" for the GPU runs
TAG = "small" if SMALL else "gpu"
N_GAL = 100_000 if SMALL else 10_000_000
GALAXY = os.path.join(RUNS, "ics", f"mw_{N_GAL}.npz")
PREP, LIVE, STATIC, FIGS = (os.path.join(RUNS, f"{x}_{TAG}") for x in ("prepare", "pal5_live", "pal5_static", "figs"))
T_GYR = 0.002 if SMALL else 2.0

def run(args, py=JAX_PY, prefix=()):
    cmd = [*prefix, py, *args]
    print("$", " ".join(cmd), flush=True)
    if RUN:
        out = subprocess.run(cmd, cwd=REPO, capture_output=True, text=True)
        print(out.stdout[-3000:])
        if out.returncode:
            print(out.stderr[-3000:])
            raise RuntimeError(f"rc={out.returncode}")
print(TAG, RUNS)

## 1. The catalogue

`tools/fetch_baumgardt.py` downloads the orbit and structure tables and records URL, date and sha256
(`SOURCE.json`). `odisseo.globular_clusters` parses them; Galactocentric coordinates come from astropy's
`Galactocentric` frame (its defaults; `BAUMGARDT_FRAME` reproduces the catalogue's own columns).

In [ ]:
if not os.path.exists(os.path.join(DATA, "SOURCE.json")):
    run(["tools/fetch_baumgardt.py", "--out", DATA])
sys.path.insert(0, REPO)
from odisseo import globular_clusters as gc
cat = gc.load_baumgardt(directory=DATA)
for row in gc.cluster_table(["Pal 5", "NGC 5466", "M68"], cat):
    print(f"{row['name']:9s} x = {row['x'] * 10} kpc, M = {row['mass_msun']:.3g} Msun, "
          f"r_h = {row['r_half_pc']} pc, W0 = {row['w0']:.2f}")

## 2. The live Milky Way

The a4 `galaxy` model of `tools/a4_galaxy_ics.py` (agama self-consistent: exponential disc 6e10 Msun,
R_d 2.4 kpc; Hernquist bulge 1.2e10 Msun, a = 0.8 kpc; NFW halo 1e12 Msun, r_s 20 kpc, cut off at
100 kpc), seed 7, with its potential exported for step 3.

In [ ]:
os.makedirs(os.path.dirname(GALAXY), exist_ok=True)
if not os.path.exists(GALAXY):
    run(["tools/a4_galaxy_ics.py", "galaxy", "--output", GALAXY, "--n-particles", str(N_GAL), "--seed", "7",
         "--export-potential", GALAXY.replace(".npz", ".pot.ini")], py=AGAMA_PY)

## 3. The static potential and the orbit back in time

`prepare.py` fits Odisseo's static MW (NFW + MN3 disc + Hernquist bulge) to the live model's potential where
the orbit goes, and integrates Pal 5's present position back by `T_GYR` in the live model's potential.

In [ ]:
run(["notebooks/gc_in_mw/prepare.py", "--potential", GALAXY.replace(".npz", ".pot.ini"), "--names", "Pal 5",
     "--t-gyr", str(T_GYR), "--out", PREP], py=AGAMA_PY)
prep = json.load(open(os.path.join(PREP, "prepare.json")))
print(json.dumps({k: prep[k] for k in ("fit_params", "fit_stats", "vc_kms_at_8kpc",
                                        "present_offset_static_vs_live_kpc")}, indent=1))

## 4. The runs

Live: `run_live.py` (ProductionRunner + ClustersRunnerScheme, one card). Static: `run_static.py`
(Odisseo's direct-sum leapfrog in the fitted static potential). Same stars (King model, W0 from the
catalogue's concentration, seed 0).

In [ ]:
small = ["--small"] if SMALL else []
run(["notebooks/gc_in_mw/run_live.py", "--galaxy", GALAXY, "--prepare", PREP, "--M", "4096", "--out", LIVE,
     *small], prefix=PREFIX)
run(["notebooks/gc_in_mw/run_static.py", "--prepare", PREP, "--M", "4096", "--out", STATIC, *small],
    prefix=PREFIX)

## 5. Conservation, stripping and the stream

In [ ]:
run(["notebooks/gc_in_mw/analyse.py", "--prepare", PREP, "--live", LIVE, "--static", STATIC, "--out", FIGS],
    py=AGAMA_PY)
summary = json.load(open(os.path.join(FIGS, "summary.json")))
print(json.dumps({k: summary[k] for k in ("live", "progenitor_offset_live_vs_static_kpc",
                                           "point_orbit_offset_live_model_vs_static_kpc", "track")}, indent=1))
for f in ("potential.png", "conservation.png", "stream.png"):
    display(Image(os.path.join(FIGS, f), width=900))

## 6. The K ~ 150 bundle

Every catalogue cluster with a mass and a half-mass radius (K = 165, M = 1000 stars each) at today's
positions in the live MW: `run_live.py --names ALL`. The functional run (a few segments) and the long run
are in `notebooks/gc_in_mw/bundle.sh`; scaling numbers are HoreKa's.